# 04 — DistilBERT Prosodic Boundary Classifier

Fine-tunes `distilbert-base-uncased` on the silver-standard labels produced by
`annotation_pipeline_v4.ipynb`.

**Task (current):** Binary token classification — predict whether each word is a
prosodic phrase boundary (1) or not (0).

**Baseline to beat:** PSST text-only GPT-Neo F1 = 0.77

Disclaimer: Code was largely generated with the help of Claude Sonnet 3.5/4.6 (Anthropic, 2026 February - May). Prompts, code tweaks and verification by me.

---

## How to use this notebook

Run cells **top to bottom**. The only cell you should need to edit is **Cell 1**.

| Section | Cells | What it does |
|---|---|---|
| **1 · Configuration** | 1 | All hyperparameters, flags, and paths |
| **2 · Environment** | 2 | Mount Drive, install packages (incl. spaCy) |
| **3 · Imports** | 3 | All imports + POS vocabulary constants |
| **4 · Load Data** | 4 | Read label files from Drive |
| **4b · POS Pre-generation** | 4b | Tag all samples with spaCy; cache to Drive |
| **5 · Dataset** | 5 | `ProsodyDataset`, label alignment, POS support |
| **7 · Model** | 7 | `ProsodyBoundaryModel` with optional POS embedding |
| **8 · Metrics** | 8 | `flatten_predictions`, `compute_metrics` |
| **12 · Multi-run harness** | 12 | `run_experiment()` — train + eval one config |
| **14 · Run queue** | 14 | Define and execute all runs |

---

## POS feature modes (`TRAIN_ON_TEXT` / `TRAIN_ON_POS`)

| `TRAIN_ON_TEXT` | `TRAIN_ON_POS` | Mode |
|---|---|---|
| `True` | `False` | **Text only** — original behaviour, no POS |
| `True` | `True` | **Combined** — text + POS embedding injected post-transformer |
| `False` | `True` | **POS only** — POS abbreviations replace words as DistilBERT input |
| `False` | `False` | ❌ invalid |

POS tagging is **always** performed on the original punctuated word list regardless of `STRIP_PUNCTUATION`. Punctuation stripping (if enabled) is applied *after* tagging.

---

## Output folder structure

```
runs/
└── run_{timestamp}_{config_hash}/
    ├── checkpoint/          ← model weights + tokenizer (HuggingFace format)
    ├── hparams.json         ← ALL hyperparameters (model + annotation tool)
    ├── test_predictions.json  ← per-sample word-level predictions
    ├── curves.png           ← loss + F1 learning curves
    └── confusion_matrix.png
```


---
## Section 1 · Configuration


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  The only cell you should need to edit between runs.         ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, hashlib
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT       = "/content/drive/MyDrive/Capstone/project"
RUNS_ROOT        = f"{DRIVE_ROOT}/runs"
BATCHED_ROOT     = f"{DRIVE_ROOT}/labels/clean-100"
BATCHED_ROOT_360 = f"{DRIVE_ROOT}/labels/clean-360"
BATCHED_ROOT_SBC = f"{DRIVE_ROOT}/labels/sbcsae"

# ── Data split ────────────────────────────────────────────────────────────────
SPLIT_SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10

# ── Text pre-processing ───────────────────────────────────────────────────────
# POS tagging is ALWAYS performed on the original punctuated word list.
# STRIP_PUNCTUATION controls whether DistilBERT sees punctuation tokens during
# training — it does NOT affect what text the POS tagger receives.
STRIP_PUNCTUATION = True

# ── POS feature toggles ───────────────────────────────────────────────────────
# TRAIN_ON_TEXT : True  → DistilBERT receives the real word sequence as input.
#                 False → DistilBERT receives POS abbreviation tokens instead
#                         (e.g. "I ate cake" becomes "pro vb nn").
#                         Label alignment is unchanged — labels still correspond
#                         to word positions, just the input surface form differs.
# TRAIN_ON_POS  : True  + TRAIN_ON_TEXT=True  → combined mode: POS embeddings
#                         are added to DistilBERT's last hidden state *after*
#                         the transformer (post-transformer injection, Cell 7).
#                 True  + TRAIN_ON_TEXT=False → POS-only mode; no POS embedding
#                         layer is needed because POS IS the input text.
#                 False → POS disabled entirely; original text-only behaviour.
# Setting both False is invalid and raises an error below.
TRAIN_ON_TEXT     = True
TRAIN_ON_POS      = True
POS_EMBEDDING_DIM = 64      # embedding size before Linear projection to H=768
                             # (only used in combined mode; ignored otherwise)

EXTRA_FEATURES    = []

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 6
WARMUP_STEPS  = 0
WEIGHT_DECAY  = 0.01

# ── Class-imbalance strategy (boundary head only) ─────────────────────────────
IMBALANCE_STRATEGY   = "none"
BOUNDARY_LOSS_WEIGHT = 5.0

# ── Multi-task loss weights ───────────────────────────────────────────────────
# Weight for each head's contribution to the combined loss.
# 1.0 = equal weighting. Raise a weight to emphasise that head's gradient.
# Intonation and break_idx heads supervise only at boundary positions, so
# their raw loss magnitude will naturally be smaller than the boundary head's.
BOUNDARY_TASK_WEIGHT   = 1.0
INTONATION_TASK_WEIGHT = 1.0
BREAK_IDX_TASK_WEIGHT  = 1.0

# ── Run metadata ──────────────────────────────────────────────────────────────
RUN_NOTES = ""

# ── Validate input-modality toggles ──────────────────────────────────────────
if not TRAIN_ON_TEXT and not TRAIN_ON_POS:
    raise ValueError(
        "TRAIN_ON_TEXT=False and TRAIN_ON_POS=False is invalid: "
        "at least one input modality must be enabled."
    )

# ── Auto-generated run ID (do not edit) ──────────────────────────────────────
_cfg = dict(
    SPLIT_SEED=SPLIT_SEED, TRAIN_FRAC=TRAIN_FRAC, VAL_FRAC=VAL_FRAC,
    STRIP_PUNCTUATION=STRIP_PUNCTUATION,
    TRAIN_ON_TEXT=TRAIN_ON_TEXT, TRAIN_ON_POS=TRAIN_ON_POS,
    POS_EMBEDDING_DIM=POS_EMBEDDING_DIM,
    EXTRA_FEATURES=EXTRA_FEATURES,
    MODEL_NAME=MODEL_NAME, MAX_LENGTH=MAX_LENGTH,
    BATCH_SIZE=BATCH_SIZE, LEARNING_RATE=LEARNING_RATE,
    NUM_EPOCHS=NUM_EPOCHS, WARMUP_STEPS=WARMUP_STEPS, WEIGHT_DECAY=WEIGHT_DECAY,
    IMBALANCE_STRATEGY=IMBALANCE_STRATEGY, BOUNDARY_LOSS_WEIGHT=BOUNDARY_LOSS_WEIGHT,
    BOUNDARY_TASK_WEIGHT=BOUNDARY_TASK_WEIGHT,
    INTONATION_TASK_WEIGHT=INTONATION_TASK_WEIGHT,
    BREAK_IDX_TASK_WEIGHT=BREAK_IDX_TASK_WEIGHT,
)
_cfg_hash = hashlib.md5(json.dumps(_cfg, sort_keys=True).encode()).hexdigest()[:8]
RUN_ID    = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{_cfg_hash}"
RUN_DIR   = f"{RUNS_ROOT}/{RUN_ID}"

print("✓ Configuration loaded.")
print(f"  Run ID:               {RUN_ID}")
print(f"  Strip punctuation:    {STRIP_PUNCTUATION}")
print(f"  Train on text:        {TRAIN_ON_TEXT}")
print(f"  Train on POS:         {TRAIN_ON_POS}")
if TRAIN_ON_POS:
    _mode = "combined (text + POS embedding)" if TRAIN_ON_TEXT else "POS-only (POS as input text)"
    print(f"  POS mode:             {_mode}")
    if TRAIN_ON_TEXT:
        print(f"  POS embedding dim:    {POS_EMBEDDING_DIM} → projected to 768")
print(f"  Extra features:       {EXTRA_FEATURES or 'none'}")
print(f"  Imbalance strategy:   {IMBALANCE_STRATEGY}")
print(f"  Epochs / LR / Batch:  {NUM_EPOCHS} / {LEARNING_RATE} / {BATCH_SIZE}")
print(f"  Task weights:         boundary={BOUNDARY_TASK_WEIGHT}  "
      f"intonation={INTONATION_TASK_WEIGHT}  break_idx={BREAK_IDX_TASK_WEIGHT}")


---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + install packages                     ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

for d in [BATCHED_ROOT, RUNS_ROOT]:
    os.makedirs(d, exist_ok=True)

!pip install -q transformers datasets scikit-learn matplotlib
# spaCy is used for POS tagging in Cell 4b (only needed when TRAIN_ON_POS=True,
# but installed unconditionally to avoid surprises mid-run).
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import torch
print(f"\n✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠  No GPU — training will be slow.")
    print("   Runtime → Change runtime type → T4 GPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")


---
## Section 3 · Imports


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Imports                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, string, random, shutil, traceback
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    DistilBertPreTrainedModel,
    DistilBertConfig,
    get_linear_schedule_with_warmup,
)
from tqdm.notebook import tqdm as tqdm_nb

# JSON encoder that handles numpy scalar types produced by sklearn metrics.
class _NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

# Reproducibility
torch.manual_seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
random.seed(SPLIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SPLIT_SEED)

# ── POS tag vocabulary (Universal Dependencies / spaCy UPOS) ─────────────────
# Defined here unconditionally so ProsodyDataset and ProsodyBoundaryModel can
# reference them regardless of the TRAIN_ON_POS setting.
#
# Each UPOS tag maps to a short abbreviation. Abbreviations are chosen to be
# single WordPiece tokens in DistilBERT where possible. The first-subword rule
# (already used for label alignment) handles any that happen to split.
#
# POS tagging is performed on the *original* punctuated word list (before any
# STRIP_PUNCTUATION pass). This gives the most linguistically accurate tags —
# e.g. "Well, I don't know." tags differently from "Well I don't know".
UNIVERSAL_TO_TOKEN = {
    "ADJ":   "adj",   # adjective
    "ADP":   "adp",   # adposition (preposition / postposition)
    "ADV":   "adv",   # adverb
    "AUX":   "aux",   # auxiliary verb
    "CCONJ": "cc",    # coordinating conjunction
    "DET":   "det",   # determiner
    "INTJ":  "ij",    # interjection
    "NOUN":  "nn",    # noun
    "NUM":   "num",   # numeral
    "PART":  "pt",    # particle
    "PRON":  "pro",   # pronoun
    "PROPN": "np",    # proper noun
    "PUNCT": "pun",   # punctuation
    "SCONJ": "sc",    # subordinating conjunction
    "SYM":   "sym",   # symbol
    "VERB":  "vb",    # verb
    "X":     "xx",    # other / unknown
    "SPACE": "sp",    # whitespace token
}
UNK_POS_TOKEN = "unk"   # fallback for any tag not in the map above

# Integer IDs: 0 = PAD (used for special tokens and continuation sub-words)
_POS_TAG_NAMES = ["PAD"] + list(UNIVERSAL_TO_TOKEN.keys())
POS_TAG_TO_ID  = {tag: i for i, tag in enumerate(_POS_TAG_NAMES)}
NUM_POS_TAGS   = len(_POS_TAG_NAMES)   # 19

print("✓ Imports complete.")
print(f"  POS vocabulary: {NUM_POS_TAGS} tags  (0=PAD, 1–{NUM_POS_TAGS-1}=UPOS)")


---
## Section 4 · Load Label Files

Reads `boundary/`, `intonation/`, and `break_idx/` directly from Drive.

**Label file schema (from `annotation_pipeline_v4`):**

| Directory | Key used | Type | Description |
|---|---|---|---|
| `boundary/{id}.json` | `consensus` | `list[int]` 0/1 | Training target — both annotators agreed |
| `boundary/{id}.json` | `psst` | `list[int]` | Raw PSST boundary vector (provenance) |
| `boundary/{id}.json` | `wav2tobi` | `list[int]` | Raw Wav2ToBI boundary vector (provenance) |
| `boundary/{id}.json` | `status` | `list[str]` | Per-token: `agree_boundary` / `agree_none` / `psst_only` / `wav2tobi_only` |
| `intonation/{id}.json` | `labels` | `list[int]` | 0=none, 1=rising, 2=falling, 3=level, 4=unclear |
| `break_idx/{id}.json` | `labels` | `list[str]` | `"3"` / `"4"` / `""` |

Each directory also contains a `meta.json` (tool hyperparameters) that is
excluded during file enumeration and collected separately for provenance
tracking in `hparams.json`.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load label files from batched Drive files          ║
# ╚══════════════════════════════════════════════════════════════╝
from tqdm.notebook import tqdm

def load_label_files_batched(batched_root):
    """
    Load all samples from the consolidated batch files produced by Cell 3b.

    Batch file layout:
        labels_batched/
        ├── batch_0000.json   ← dict keyed by sample_id, each value has "b", "i", "x"
        ├── batch_0001.json
        └── ...

    Returns
    -------
    samples  : list[dict]  — keys: sample_id, tokens, boundary_labels,
                             intonation_labels, break_idx_labels
    meta     : dict        — empty (meta.json not included in batches;
                             load separately from LABELS_ROOT if needed)
    """
    batch_files = sorted(
        f for f in os.listdir(batched_root)
        if f.startswith("batch_") and f.endswith(".json")
    )
    if not batch_files:
        raise FileNotFoundError(
            f"No batch files found in {batched_root}.\n"
            "Run Cell 3b first to consolidate the label files."
        )
    print(f"Found {len(batch_files)} batch files in {batched_root}.")

    samples, n_skipped = [], 0
    for fname in tqdm(batch_files, desc="Loading batches", unit="batch"):
        with open(os.path.join(batched_root, fname)) as f:
            batch = json.load(f)
            for sid, data in batch.items():
              b, i, x = data["b"], data["i"], data["x"]
              n = len(b["tokens"])
              # x is null for SBCSAE — no ToBI break indices available from
              # Du Bois transcripts. Substitute empty strings so the loader
              # and dataset produce the correct -100 masks downstream.
              x_labels = x["labels"] if x is not None else [""] * n
              if not (len(b["consensus"]) == len(i["labels"]) == len(x_labels) == n):
                  print(f"  ⚠ {sid}: mismatched label lengths — skipping.")
                  n_skipped += 1
                  continue
              samples.append({
                  "sample_id":         sid,
                  "tokens":            b["tokens"],
                  "boundary_labels":   b["consensus"],
                  "intonation_labels": i["labels"],
                  "break_idx_labels":  x_labels,
              })

    print(f"\n✓ Loaded {len(samples):,} samples  ({n_skipped} skipped).")
    if samples:
        total_tokens     = sum(len(s["tokens"])          for s in samples)
        total_boundaries = sum(sum(s["boundary_labels"]) for s in samples)
        pos_rate         = total_boundaries / max(total_tokens, 1)
        ratio            = (1 - pos_rate) / max(pos_rate, 1e-9)
        print(f"  Total words:     {total_tokens:,}")
        print(f"  Boundary tokens: {total_boundaries:,}  ({100*pos_rate:.1f}% positive rate)")
        print(f"  Non-boundary:boundary ratio ≈ {ratio:.1f}:1")
        print(f"  → Suggested BOUNDARY_LOSS_WEIGHT: ~{round(ratio)}.0")

    return samples, {}


all_samples, label_meta = load_label_files_batched(BATCHED_ROOT)


---
## Section 4b · POS Pre-generation & Enrichment

Runs spaCy on all loaded samples to generate part-of-speech tags. Results are
cached to `labels/{subset}/pos/batch_XXXX.json` on Drive and reused on
subsequent runs — re-running this cell is cheap (files are skipped if they
already exist).

**This cell is a no-op when `TRAIN_ON_POS=False`.**

POS files mirror the batch file structure one directory below:
```
labels/clean-100/
├── batch_0000.json       ← label batch (existing)
└── pos/
    └── batch_0000.json   ← { sample_id: ["NOUN", "VERB", ...], ... }
```

spaCy's forced tokenization (`Doc(vocab, words=words)`) is used to guarantee
1:1 word/tag alignment with our word list, bypassing spaCy's own tokenizer.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4b — POS pre-generation & enrichment                   ║
# ║  Generates labels/{subset}/pos/batch_XXXX.json files.        ║
# ║  Skips files that already exist.  No-op if TRAIN_ON_POS=False║
# ╚══════════════════════════════════════════════════════════════╝

# ── Global spaCy model handle (set below when TRAIN_ON_POS=True) ─────────────
_nlp_pos = None


def _generate_pos_files(batched_root):
    """
    For every batch_XXXX.json in batched_root, write a corresponding POS file
    to batched_root/pos/batch_XXXX.json.  Skips files that already exist.
    Schema: { sample_id: [UPOS_string, ...] }  — one tag per word, word-level.

    Always-safe to call: returns immediately if TRAIN_ON_POS=False or if
    _nlp_pos has not been loaded yet.
    """
    if not TRAIN_ON_POS:
        return
    if _nlp_pos is None:
        raise RuntimeError(
            "_nlp_pos is not loaded.  Set TRAIN_ON_POS=True and re-run Cell 4b."
        )
    from spacy.tokens import Doc

    pos_root = os.path.join(batched_root, "pos")
    os.makedirs(pos_root, exist_ok=True)

    batch_files = sorted(
        f for f in os.listdir(batched_root)
        if f.startswith("batch_") and f.endswith(".json")
    )
    if not batch_files:
        print(f"  ⚠ No batch files in {batched_root} — nothing to tag.")
        return

    n_new = 0
    for fname in tqdm_nb(batch_files, desc=f"POS tagging {os.path.basename(batched_root)}"):
        pos_path = os.path.join(pos_root, fname)
        if os.path.exists(pos_path):
            continue   # already done — skip

        with open(os.path.join(batched_root, fname)) as f:
            batch = json.load(f)

        pos_batch = {}
        for sid, data in batch.items():
            words = data["b"]["tokens"]
            if not words:
                pos_batch[sid] = []
                continue
            # Forced tokenization: Doc(vocab, words=words) sets exactly len(words)
            # tokens without running spaCy's tokenizer, guaranteeing 1:1 alignment.
            spaces = [True] * (len(words) - 1) + [False]
            doc = Doc(_nlp_pos.vocab, words=words, spaces=spaces)
            for pipe_name in ("tok2vec", "tagger"):
                if _nlp_pos.has_pipe(pipe_name):
                    _nlp_pos.get_pipe(pipe_name)(doc)
            pos_batch[sid] = [t.pos_ or "X" for t in doc]

        with open(pos_path, "w") as f:
            json.dump(pos_batch, f)
        n_new += 1

    total = len(batch_files)
    print(f"  ✓ {n_new} new POS files written, {total - n_new} already existed "
          f"({total} total in {os.path.basename(batched_root)}).")


def _load_pos_for_samples(samples, batched_root):
    """
    Enrich each sample dict in-place with a 'pos_tags' key (list[str] of UPOS
    tags, word-level, same length as 'tokens').
    Reads from batched_root/pos/*.json.  Falls back to ['X']*n for missing samples.
    No-op if TRAIN_ON_POS=False.
    """
    if not TRAIN_ON_POS:
        return

    pos_root = os.path.join(batched_root, "pos")
    # Merge all POS batch files into a single lookup dict
    pos_cache = {}
    for fname in sorted(f for f in os.listdir(pos_root) if f.endswith(".json")):
        with open(os.path.join(pos_root, fname)) as f:
            pos_cache.update(json.load(f))

    n_missing = 0
    for s in samples:
        tags = pos_cache.get(s["sample_id"])
        if tags is not None and len(tags) == len(s["tokens"]):
            s["pos_tags"] = tags
        else:
            # Fallback: tag as unknown.  Logged below.
            s["pos_tags"] = ["X"] * len(s["tokens"])
            n_missing += 1

    if n_missing:
        print(f"  ⚠ {n_missing}/{len(samples)} samples fell back to 'X' "
              f"(missing from cache or length mismatch).")
    else:
        print(f"  ✓ POS loaded for all {len(samples):,} samples "
              f"from {os.path.basename(batched_root)}.")


# ── Execute (clean-100 only; c360 and SBC are handled in Cell 14) ─────────────
if TRAIN_ON_POS:
    import spacy
    from spacy.tokens import Doc

    print("Loading spaCy en_core_web_sm ...")
    _nlp_pos = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])
    print(f"  ✓ spaCy loaded  (active pipes: {_nlp_pos.pipe_names})")

    print("\nGenerating POS files for clean-100 ...")
    _generate_pos_files(BATCHED_ROOT)
    print("\nLoading POS into all_samples (clean-100) ...")
    _load_pos_for_samples(all_samples, BATCHED_ROOT)
    print("\n✓ Cell 4b complete.  c360 and SBC will be enriched in Cell 14.")
else:
    print("TRAIN_ON_POS=False — Cell 4b skipped.")
    print("  (_generate_pos_files and _load_pos_for_samples are defined and ready)")
    print("  (Set TRAIN_ON_POS=True in Cell 1 and re-run to enable POS features.)")


---
## Section 5 · Dataset

`ProsodyDataset` tokenizes each sentence with DistilBERT and aligns word-level
boundary labels to sub-word tokens using the standard HuggingFace
`tokenize_and_align_labels` pattern:

- **First sub-word** of word *i* → receives the word's label
- **Continuation sub-words** → label set to `-100` (ignored by loss)
- **Special tokens** (`[CLS]`, `[SEP]`, `[PAD]`) → `-100`

**POS-mode behaviour:**

| `train_on_text` | `train_on_pos` | Input to DistilBERT | `pos_ids` tensor? |
|---|---|---|---|
| True | False | original words | no |
| True | True | original words | yes — aligned to sub-words |
| False | True | POS abbreviations (e.g. `"nn vb nn"`) | no |


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — ProsodyDataset + collate_fn                        ║
# ╚══════════════════════════════════════════════════════════════╝

PUNCT_SET = set(string.punctuation)

def is_punct_only(word):
    """True if every character in `word` is ASCII punctuation."""
    return bool(word) and all(c in PUNCT_SET for c in word)


class ProsodyDataset(Dataset):
    """
    Token-classification dataset for prosodic boundary detection.

    Label alignment rules
    ─────────────────────
    All three heads share the same rule: first sub-word of word i receives
    the word's label; continuation sub-words and special tokens get -100.

    Boundary   : 0 / 1  — all positions supervised
    Intonation : rising(1)→0 | falling(2)→1 | level(3)→2
                 none(0) and unclear(4) → -100  (masked)
    Break index: "3"→0 | "4"→1
                 ""  and any missing value → -100 (masked; "" means Wav2ToBI
                 was never asked, not an affirmative "no break")

    POS modes
    ─────────
    train_on_text=True,  train_on_pos=False : original text; no POS tensor
    train_on_text=True,  train_on_pos=True  : original text + pos_ids tensor
                                              (for post-transformer injection)
    train_on_text=False, train_on_pos=True  : POS abbreviation strings as input
                                              text; no pos_ids tensor needed
    """

    SPK_CHANGE_TOKEN = "/"   # must match the pipeline constant

    def __init__(self, samples, tokenizer, max_length=128,
                 strip_punctuation=False,
                 train_on_text=True,
                 train_on_pos=False,
                 extra_features=None):
        self.tokenizer         = tokenizer
        self.max_length        = max_length
        self.strip_punctuation = strip_punctuation
        self.train_on_text     = train_on_text
        self.train_on_pos      = train_on_pos
        self.extra_features    = extra_features or []
        self.items             = [self._process(s) for s in samples]

    def _process(self, sample):
        words    = list(sample["tokens"])
        b_labels = list(sample["boundary_labels"])
        i_labels = list(sample.get("intonation_labels", []))
        x_labels = list(sample.get("break_idx_labels",  []))
        # Word-level UPOS tags.  Present if sample was enriched by Cell 4b.
        # Defaults to "X" (unknown) if missing — safe fallback.
        pos_tags = list(sample.get("pos_tags", ["X"] * len(words)))

        # ── Optional: strip punctuation-only tokens ───────────────────────
        # POS tagging was done on the ORIGINAL word list (before this step),
        # so tag quality is unaffected by STRIP_PUNCTUATION.
        if self.strip_punctuation:
            quints = [
                (w, b, i, x, p)
                for w, b, i, x, p in zip(words, b_labels, i_labels, x_labels, pos_tags)
                if not is_punct_only(w) or w == self.SPK_CHANGE_TOKEN
            ]
            if quints:
                words, b_labels, i_labels, x_labels, pos_tags = map(list, zip(*quints))
            else:
                words, b_labels, i_labels, x_labels, pos_tags = [], [], [], [], []

        # ── Choose input sequence: real words or POS abbreviations ────────
        # POS-only mode (train_on_text=False): replace each word with its
        # POS abbreviation token (e.g. "NOUN"→"nn", "VERB"→"vb").
        # DistilBERT reads a sequence like "pro vb nn det nn" instead of
        # the actual words.  Label alignment is unchanged — the first subword
        # of each POS token maps to the same boundary label as the original word.
        if self.train_on_text:
            input_words = words
        else:
            # train_on_pos must be True here (validated in Cell 1)
            input_words = [UNIVERSAL_TO_TOKEN.get(p, UNK_POS_TOKEN) for p in pos_tags]

        # ── Sub-word tokenization ─────────────────────────────────────────
        encoding = self.tokenizer(
            input_words,
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)

        # ── Align boundary labels ─────────────────────────────────────────
        aligned_boundary = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_boundary.append(-100)
            elif word_i != prev_word_i:
                aligned_boundary.append(
                    b_labels[word_i] if word_i < len(b_labels) else -100)
            else:
                aligned_boundary.append(-100)
            prev_word_i = word_i

        # ── Speaker-change position mask ──────────────────────────────────
        aligned_spk_mask = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_spk_mask.append(False)
            elif word_i != prev_word_i:
                aligned_spk_mask.append(words[word_i] == self.SPK_CHANGE_TOKEN)
            else:
                aligned_spk_mask.append(False)
            prev_word_i = word_i

        # ── Align intonation labels ───────────────────────────────────────
        # Raw: 0=none, 1=rising, 2=falling, 3=level, 4=unclear
        # Mapped: rising→0, falling→1, level→2; none+unclear → -100
        _INTONATION_MAP = {1: 0, 2: 1, 3: 2}
        aligned_intonation = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_intonation.append(-100)
            elif word_i != prev_word_i:
                raw = i_labels[word_i] if word_i < len(i_labels) else -1
                aligned_intonation.append(_INTONATION_MAP.get(raw, -100))
            else:
                aligned_intonation.append(-100)
            prev_word_i = word_i

        # ── Align break index labels ──────────────────────────────────────
        # Raw: "" (unannotated), "3" (intermediate phrase), "4" (intonational phrase)
        # "" is masked (not class-0) because absence of annotation ≠ negative example.
        _BREAK_IDX_MAP = {"3": 0, "4": 1}
        aligned_break_idx = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_break_idx.append(-100)
            elif word_i != prev_word_i:
                raw = x_labels[word_i] if word_i < len(x_labels) else ""
                aligned_break_idx.append(_BREAK_IDX_MAP.get(raw, -100))
            else:
                aligned_break_idx.append(-100)
            prev_word_i = word_i

        # ── Build POS ID tensor (combined mode only) ──────────────────────
        # Only when train_on_text=True AND train_on_pos=True.
        # PAD (index 0) at [CLS]/[SEP]/[PAD] and continuation sub-words.
        # These IDs are looked up in ProsodyBoundaryModel.pos_embedding and
        # the result is projected + added to DistilBERT's last hidden state
        # (post-transformer injection — see Cell 7 for design rationale).
        extra = {}
        if self.train_on_pos and self.train_on_text:
            pos_ids_list = []
            prev_word_i = None
            for word_i in word_ids:
                if word_i is None:
                    pos_ids_list.append(POS_TAG_TO_ID["PAD"])
                elif word_i != prev_word_i:
                    tag = pos_tags[word_i] if word_i < len(pos_tags) else "X"
                    # Fall back to "X" ID if tag somehow not in vocab
                    pos_ids_list.append(
                        POS_TAG_TO_ID.get(tag, POS_TAG_TO_ID.get("X", 0))
                    )
                else:
                    pos_ids_list.append(POS_TAG_TO_ID["PAD"])
                prev_word_i = word_i
            extra["pos_ids"] = torch.tensor(pos_ids_list, dtype=torch.long)

        return {
            "sample_id":         sample["sample_id"],
            "tokens":            words,   # post-strip; aligned with label tensors
            "input_ids":         encoding["input_ids"].squeeze(0),
            "attention_mask":    encoding["attention_mask"].squeeze(0),
            "spk_mask":          torch.tensor(aligned_spk_mask, dtype=torch.bool),
            "labels":            torch.tensor(aligned_boundary,   dtype=torch.long),
            "intonation_labels": torch.tensor(aligned_intonation, dtype=torch.long),
            "break_idx_labels":  torch.tensor(aligned_break_idx,  dtype=torch.long),
            **extra,
        }

    def __len__(self):  return len(self.items)
    def __getitem__(self, idx): return self.items[idx]


def prosody_collate_fn(batch):
    """Custom collate: stack tensors, keep sample_ids + tokens as plain lists."""
    out = {
        "input_ids":         torch.stack([b["input_ids"]         for b in batch]),
        "attention_mask":    torch.stack([b["attention_mask"]    for b in batch]),
        "spk_mask":          torch.stack([b["spk_mask"]          for b in batch]),
        "labels":            torch.stack([b["labels"]            for b in batch]),
        "intonation_labels": torch.stack([b["intonation_labels"] for b in batch]),
        "break_idx_labels":  torch.stack([b["break_idx_labels"]  for b in batch]),
        "sample_ids":        [b["sample_id"] for b in batch],
        "tokens":            [b["tokens"]    for b in batch],
    }
    # pos_ids is only present in combined mode (train_on_text=True, train_on_pos=True)
    if "pos_ids" in batch[0]:
        out["pos_ids"] = torch.stack([b["pos_ids"] for b in batch])
    return out


print("✓ ProsodyDataset and prosody_collate_fn defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — ProsodyBoundaryModel                               ║
# ╚══════════════════════════════════════════════════════════════╝

class ProsodyBoundaryModel(DistilBertPreTrainedModel):
    """
    Architecture
    ────────────
    DistilBERT encoder
        [+ optional POS embedding addition, post-transformer]
        └─► dropout (seq_classif_dropout)
             ├─► boundary_head    Linear(H → 2)   all positions
             ├─► intonation_head  Linear(H → 3)   rising / falling / level
             └─► break_idx_head   Linear(H → 2)   index-3 / index-4

    POS embedding design (combined mode)
    ─────────────────────────────────────
    When use_pos_embedding=True, a small nn.Embedding(NUM_POS_TAGS, pos_emb_dim)
    maps integer POS IDs to vectors of size pos_emb_dim (default 64).  A Linear
    projection then maps these to H=768, and the result is ADDED to DistilBERT's
    last hidden state AFTER the transformer.

    Rationale for post-transformer injection:
      Adding to input embeddings (pre-transformer) would disturb DistilBERT's
      pretrained representations, which were never trained with POS tokens.
      Post-transformer injection leaves DistilBERT's pretrained weights fully
      intact and makes the POS contribution easily ablatable.
      If you want to experiment with pre-transformer injection, move the pos_emb
      addition into DistilBertEmbeddings or create a custom input embedding layer.

    Each head returns logits over all (B, T) positions; masking to the
    relevant subset is handled by CrossEntropyLoss(ignore_index=-100).
    Loss combination weights are controlled by BOUNDARY/INTONATION/
    BREAK_IDX_TASK_WEIGHT in Cell 1 and applied in the training loop.
    """

    def __init__(self, config):
        super().__init__(config)
        self.distilbert  = DistilBertModel(config)
        self.dropout     = nn.Dropout(config.seq_classif_dropout)

        # ── Optional POS embedding (combined mode only) ───────────────────
        # Controlled by config.use_pos_embedding (set in build_model / run_experiment).
        # In POS-only mode (TRAIN_ON_TEXT=False), this is NOT used — POS abbreviations
        # are the text input itself, so no separate embedding injection is needed.
        self.use_pos_embedding = getattr(config, "use_pos_embedding", False)
        if self.use_pos_embedding:
            _pos_emb_dim  = getattr(config, "pos_emb_dim",  64)
            _num_pos_tags = getattr(config, "num_pos_tags", NUM_POS_TAGS)
            self.pos_embedding = nn.Embedding(
                _num_pos_tags, _pos_emb_dim, padding_idx=0
            )
            # Project from pos_emb_dim (e.g. 64) up to DistilBERT's hidden dim (768).
            # bias=False keeps the projection as a pure linear scaling; a bias term
            # would shift every position's representation equally, which is not useful.
            self.pos_proj = nn.Linear(_pos_emb_dim, config.hidden_size, bias=False)

        # ── Classification heads ──────────────────────────────────────────
        self.boundary_head   = nn.Linear(config.hidden_size, 2)
        self.intonation_head = nn.Linear(config.hidden_size, 3)  # rising/falling/level
        self.break_idx_head  = nn.Linear(config.hidden_size, 2)  # index-3/index-4
        self.post_init()

    def forward(self, input_ids, attention_mask, pos_ids=None,
                labels=None, intonation_labels=None, break_idx_labels=None,
                loss_weights=None):
        """
        Parameters
        ----------
        input_ids          : (B, T)
        attention_mask     : (B, T)
        pos_ids            : (B, T) LongTensor | None — POS tag IDs aligned to sub-words
                             Only passed in combined mode (train_on_text=True + train_on_pos=True)
        labels             : (B, T)  boundary targets; -100 at ignored positions
        intonation_labels  : (B, T)  0/1/2 at boundary positions, -100 elsewhere
        break_idx_labels   : (B, T)  0/1   at boundary positions, -100 elsewhere
        loss_weights       : (2,) FloatTensor | None — boundary class weights only

        Returns
        -------
        dict with keys:
            losses             : dict  {head_name: scalar loss tensor}  (when labels given)
            boundary_logits    : (B, T, 2)
            intonation_logits  : (B, T, 3)
            break_idx_logits   : (B, T, 2)
        """
        outputs = self.distilbert(input_ids=input_ids,
                                  attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)   # (B, T, H)

        # ── POS injection (post-transformer, combined mode only) ──────────
        # The POS embedding is projected to H=768 and added element-wise to
        # seq_out.  Padding positions (pos_ids == 0) contribute a near-zero
        # vector because padding_idx=0 in the Embedding layer.
        #
        # This block is skipped entirely when:
        #   - use_pos_embedding=False  (text-only mode), or
        #   - pos_ids is None          (POS-only mode or inference without POS)
        #
        # To move to pre-transformer injection: apply pos_embedding + pos_proj
        # to the token embeddings BEFORE calling self.distilbert(), then sum.
        if self.use_pos_embedding and pos_ids is not None:
            pos_emb = self.pos_proj(self.pos_embedding(pos_ids))   # (B, T, H)
            seq_out = seq_out + pos_emb

        boundary_logits   = self.boundary_head(seq_out)     # (B, T, 2)
        intonation_logits = self.intonation_head(seq_out)   # (B, T, 3)
        break_idx_logits  = self.break_idx_head(seq_out)    # (B, T, 2)

        losses = {}
        if labels is not None:
            losses["boundary"] = nn.CrossEntropyLoss(
                weight=loss_weights, ignore_index=-100)(
                boundary_logits.view(-1, 2), labels.view(-1))

        if intonation_labels is not None and (intonation_labels != -100).any():
            losses["intonation"] = nn.CrossEntropyLoss(ignore_index=-100)(
                intonation_logits.view(-1, 3), intonation_labels.view(-1))

        if break_idx_labels is not None and (break_idx_labels != -100).any():
            losses["break_idx"] = nn.CrossEntropyLoss(ignore_index=-100)(
                break_idx_logits.view(-1, 2), break_idx_labels.view(-1))

        out = {
            "boundary_logits":   boundary_logits,
            "intonation_logits": intonation_logits,
            "break_idx_logits":  break_idx_logits,
        }
        if losses:
            out["losses"] = losses
        return out



    @classmethod
    def _can_set_experts_implementation(cls):
        return False


def build_model(use_pos_embedding=False, pos_emb_dim=64):
    """Instantiate ProsodyBoundaryModel from pretrained DistilBERT weights."""
    cfg = DistilBertConfig.from_pretrained(MODEL_NAME,
                                           num_labels=2,
                                           seq_classif_dropout=0.2)
    # Store POS-related config so it's serialised with save_pretrained()
    cfg.use_pos_embedding = use_pos_embedding
    cfg.pos_emb_dim       = pos_emb_dim
    cfg.num_pos_tags      = NUM_POS_TAGS
    model = ProsodyBoundaryModel.from_pretrained(MODEL_NAME, config=cfg,
                                                  ignore_mismatched_sizes=True,
                                                  _fast_init=False)
    model.gradient_checkpointing_enable()
    return model.to(device)


model = build_model(
    use_pos_embedding=(TRAIN_ON_TEXT and TRAIN_ON_POS),
    pos_emb_dim=POS_EMBEDDING_DIM,
)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ ProsodyBoundaryModel ready  ({n_params:,} trainable parameters)")
print(f"  POS embedding:  {TRAIN_ON_TEXT and TRAIN_ON_POS}")
print(f"  Heads: boundary(2)  intonation(3)  break_idx(2)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Metrics helpers                                    ║
# ╚══════════════════════════════════════════════════════════════╝

def flatten_predictions(all_logits, all_labels, all_spk_masks=None):
    """
    Convert batched (logits, labels) to flat numpy arrays,
    discarding all positions where label == -100.
    """
    flat_preds, flat_labels = [], []
    for idx, (logits_batch, labels_batch) in enumerate(zip(all_logits, all_labels)):
        preds_batch = logits_batch.argmax(dim=-1)
        spk_batch   = all_spk_masks[idx] if all_spk_masks else None
        for b_i, (preds_seq, labels_seq) in enumerate(zip(preds_batch, labels_batch)):
            mask = labels_seq != -100
            if spk_batch is not None:
                mask = mask & ~spk_batch[b_i].to(mask.device)
            flat_preds.extend(preds_seq[mask].cpu().tolist())
            flat_labels.extend(labels_seq[mask].cpu().tolist())
    return np.array(flat_preds), np.array(flat_labels)

def compute_metrics(flat_preds, flat_labels):
    """Binary P/R/F1 for the boundary class (pos_label=1)."""
    return {
        "f1":        f1_score(flat_labels,        flat_preds, pos_label=1, zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, pos_label=1, zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, pos_label=1, zero_division=0),
    }


def compute_multiclass_metrics(flat_preds, flat_labels):
    """Macro-averaged P/R/F1 for multi-class heads (intonation, break_idx)."""
    return {
        "f1":        f1_score(flat_labels,        flat_preds, average="macro", zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, average="macro", zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, average="macro", zero_division=0),
    }


print("✓ Metrics helpers defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Multi-run harness                                 ║
# ║  Define run_experiment() — called by Cell 14 for each run.   ║
# ║  Cells 1–8 must already have been run (all_samples loaded).  ║
# ╚══════════════════════════════════════════════════════════════╝

import gc, hashlib, subprocess
from datetime import datetime

def run_experiment(overrides: dict, samples_override=None):
    """
    Train, evaluate, and save one DistilBERT run.

    Parameters
    ----------
    overrides : dict
        Any subset of the Cell-1 config keys. Unspecified keys fall back
        to the values already set in Cell 1 (the current globals).
    samples_override : list[dict] | None
        If provided, replaces all_samples for this run only.
    """

    # ── 1. Resolve config for this run ────────────────────────────────────────
    run_cfg = dict(
        SPLIT_SEED           = SPLIT_SEED,
        TRAIN_FRAC           = TRAIN_FRAC,
        VAL_FRAC             = VAL_FRAC,
        STRIP_PUNCTUATION    = STRIP_PUNCTUATION,
        TRAIN_ON_TEXT        = TRAIN_ON_TEXT,
        TRAIN_ON_POS         = TRAIN_ON_POS,
        POS_EMBEDDING_DIM    = POS_EMBEDDING_DIM,
        EXTRA_FEATURES       = EXTRA_FEATURES,
        MODEL_NAME           = MODEL_NAME,
        MAX_LENGTH           = MAX_LENGTH,
        BATCH_SIZE           = BATCH_SIZE,
        LEARNING_RATE        = LEARNING_RATE,
        NUM_EPOCHS           = NUM_EPOCHS,
        WARMUP_STEPS         = WARMUP_STEPS,
        WEIGHT_DECAY         = WEIGHT_DECAY,
        IMBALANCE_STRATEGY   = IMBALANCE_STRATEGY,
        BOUNDARY_LOSS_WEIGHT = BOUNDARY_LOSS_WEIGHT,
        BOUNDARY_TASK_WEIGHT     = BOUNDARY_TASK_WEIGHT,
        INTONATION_TASK_WEIGHT   = INTONATION_TASK_WEIGHT,
        BREAK_IDX_TASK_WEIGHT    = BREAK_IDX_TASK_WEIGHT,
        RUN_NOTES            = RUN_NOTES,
    )
    run_cfg.update(overrides)

    _SPLIT_SEED      = run_cfg["SPLIT_SEED"]
    _TRAIN_FRAC      = run_cfg["TRAIN_FRAC"]
    _VAL_FRAC        = run_cfg["VAL_FRAC"]
    _STRIP_PUNCT     = run_cfg["STRIP_PUNCTUATION"]
    _TRAIN_ON_TEXT   = run_cfg["TRAIN_ON_TEXT"]
    _TRAIN_ON_POS    = run_cfg["TRAIN_ON_POS"]
    _POS_EMB_DIM     = run_cfg["POS_EMBEDDING_DIM"]
    _EXTRA_FEATURES  = run_cfg["EXTRA_FEATURES"]
    _MODEL_NAME      = run_cfg["MODEL_NAME"]
    _MAX_LENGTH      = run_cfg["MAX_LENGTH"]
    _BATCH_SIZE      = run_cfg["BATCH_SIZE"]
    _LR              = run_cfg["LEARNING_RATE"]
    _EPOCHS          = run_cfg["NUM_EPOCHS"]
    _WARMUP          = run_cfg["WARMUP_STEPS"]
    _WD              = run_cfg["WEIGHT_DECAY"]
    _IMBALANCE       = run_cfg["IMBALANCE_STRATEGY"]
    _BLW             = run_cfg["BOUNDARY_LOSS_WEIGHT"]
    _BTW             = run_cfg["BOUNDARY_TASK_WEIGHT"]
    _ITW             = run_cfg["INTONATION_TASK_WEIGHT"]
    _XTW             = run_cfg["BREAK_IDX_TASK_WEIGHT"]
    _NOTES           = run_cfg["RUN_NOTES"]

    # Validate modality flags
    if not _TRAIN_ON_TEXT and not _TRAIN_ON_POS:
        raise ValueError(
            f"Run '{_NOTES}': TRAIN_ON_TEXT=False and TRAIN_ON_POS=False is invalid."
        )

    _cfg_hash = hashlib.md5(
        json.dumps({k: v for k, v in run_cfg.items() if k != "RUN_NOTES"},
                   sort_keys=True).encode()
    ).hexdigest()[:8]
    run_id  = _NOTES if _NOTES else f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{_cfg_hash}"
    run_dir = f"{RUNS_ROOT}/{run_id}"

    _use_pos_emb = _TRAIN_ON_TEXT and _TRAIN_ON_POS
    if _TRAIN_ON_POS:
        _pos_mode = "combined (text+POS embed)" if _TRAIN_ON_TEXT else "POS-only"
    else:
        _pos_mode = "text-only"

    print("\n" + "█" * 70)
    print(f"  RUN: {run_id}")
    print(f"  Notes:       {_NOTES or '—'}")
    print(f"  Strip punct: {_STRIP_PUNCT}  |  POS mode: {_pos_mode}")
    print(f"  Imbalance:   {_IMBALANCE}  (weight={_BLW})")
    print(f"  LR / Epochs: {_LR} / {_EPOCHS}")
    print("█" * 70)

    # ── 2. Resolve samples ────────────────────────────────────────────────────
    _samples = samples_override if samples_override is not None else all_samples
    print(f"  Data source: {len(_samples):,} samples")

    # ── 3. Split ──────────────────────────────────────────────────────────────
    _sbc_test = [s for s in _samples
                 if s["sample_id"].startswith("SBC")
                 and int(s["sample_id"][3:6]) <= 5]

    if _sbc_test:
        _sbc_test_ids = {s["sample_id"] for s in _sbc_test}
        _train_val    = [s for s in _samples if s["sample_id"] not in _sbc_test_ids]
        _test_items   = _sbc_test
        _n_tv         = len(_train_val)
        _n_train      = int(_n_tv * (_TRAIN_FRAC / (_TRAIN_FRAC + _VAL_FRAC)))
        _rng          = torch.Generator().manual_seed(_SPLIT_SEED)
        _idx          = torch.randperm(_n_tv, generator=_rng).tolist()
        _train_items  = [_train_val[i] for i in _idx[:_n_train]]
        _val_items    = [_train_val[i] for i in _idx[_n_train:]]
        print(f"  Split mode: SBC001-005 held-out test set")
    else:
        _n            = len(_samples)
        _n_train      = int(_n * _TRAIN_FRAC)
        _n_val        = int(_n * _VAL_FRAC)
        _rng          = torch.Generator().manual_seed(_SPLIT_SEED)
        _idx          = torch.randperm(_n, generator=_rng).tolist()
        _train_items  = [_samples[i] for i in _idx[:_n_train]]
        _val_items    = [_samples[i] for i in _idx[_n_train:_n_train + _n_val]]
        _test_items   = [_samples[i] for i in _idx[_n_train + _n_val:]]
        print(f"  Split mode: random 80/10/10")

    print(f"  train={len(_train_items):,}  val={len(_val_items):,}  test={len(_test_items):,}")

    # ── 4. Build datasets ─────────────────────────────────────────────────────
    _tokenizer = DistilBertTokenizerFast.from_pretrained(_MODEL_NAME)
    _ds_kwargs = dict(
        max_length        = _MAX_LENGTH,
        strip_punctuation = _STRIP_PUNCT,
        train_on_text     = _TRAIN_ON_TEXT,
        train_on_pos      = _TRAIN_ON_POS,
        extra_features    = _EXTRA_FEATURES,
    )
    _train_ds = ProsodyDataset(_train_items, _tokenizer, **_ds_kwargs)
    _val_ds   = ProsodyDataset(_val_items,   _tokenizer, **_ds_kwargs)
    _test_ds  = ProsodyDataset(_test_items,  _tokenizer, **_ds_kwargs)
    print(f"✓ Datasets built.")

    _train_loader = DataLoader(_train_ds, batch_size=_BATCH_SIZE,
                               shuffle=True,  collate_fn=prosody_collate_fn)
    _val_loader   = DataLoader(_val_ds,   batch_size=_BATCH_SIZE,
                               shuffle=False, collate_fn=prosody_collate_fn)
    _test_loader  = DataLoader(_test_ds,  batch_size=_BATCH_SIZE,
                               shuffle=False, collate_fn=prosody_collate_fn)

    # ── 5. Model ──────────────────────────────────────────────────────────────
    _cfg_model = DistilBertConfig.from_pretrained(_MODEL_NAME,
                                                   num_labels=2,
                                                   seq_classif_dropout=0.2)
    _cfg_model.use_pos_embedding = _use_pos_emb
    _cfg_model.pos_emb_dim       = _POS_EMB_DIM
    _cfg_model.num_pos_tags      = NUM_POS_TAGS
    _model = ProsodyBoundaryModel.from_pretrained(_MODEL_NAME,
                                                   config=_cfg_model,
                                                   ignore_mismatched_sizes=True,
                                                   _fast_init=False)
    _model.gradient_checkpointing_enable()
    _model = _model.to(device)
    print(f"✓ Model loaded ({sum(p.numel() for p in _model.parameters() if p.requires_grad):,} params)"
          f"  POS embedding: {_use_pos_emb}")

    # ── 6. Class-imbalance weights ────────────────────────────────────────────
    if _IMBALANCE == "weighted_loss":
        _loss_weights = torch.tensor([1.0, _BLW], dtype=torch.float).to(device)
        print(f"ℹ  Weighted loss  [non-boundary=1.0, boundary={_BLW}]")
    else:
        _loss_weights = None
        print("ℹ  No class-imbalance correction")

    # ── 7. Optimizer + scheduler ──────────────────────────────────────────────
    _optimizer   = AdamW(_model.parameters(), lr=_LR, weight_decay=_WD)
    _total_steps = _EPOCHS * len(_train_loader)
    _scheduler   = get_linear_schedule_with_warmup(
        _optimizer, num_warmup_steps=_WARMUP, num_training_steps=_total_steps)

    # ── 8. Training loop ──────────────────────────────────────────────────────
    _history = {
        "train_loss": [], "val_loss": [],
        "train_boundary_f1": [],  "val_boundary_f1": [],
        "train_boundary_prec": [], "val_boundary_prec": [],
        "train_boundary_rec": [],  "val_boundary_rec": [],
        "train_intonation_f1": [], "val_intonation_f1": [],
        "train_break_idx_f1": [],  "val_break_idx_f1": [],
    }
    _best_val_f1 = -1.0
    _best_epoch  = -1
    _best_state  = None

    print(f"\nTraining: {_EPOCHS} epochs | {len(_train_loader)} steps/epoch | {_total_steps} total steps")
    print("=" * 70)

    for epoch in tqdm_nb(range(1, _EPOCHS + 1), desc=f"{run_id[:30]} epochs"):
        # ── Train ──────────────────────────────────────────────────────────
        _model.train()
        _train_loss = 0.0
        t_b_logits, t_b_labels = [], []
        t_i_logits, t_i_labels = [], []
        t_x_logits, t_x_labels = [], []
        t_b_spk_masks = []

        for batch in tqdm_nb(_train_loader, desc=f"Ep {epoch:02d} train", leave=False):
            _ids  = batch["input_ids"].to(device)
            _mask = batch["attention_mask"].to(device)
            _bl   = batch["labels"].to(device)
            _il   = batch["intonation_labels"].to(device)
            _xl   = batch["break_idx_labels"].to(device)
            # pos_ids only present in combined mode (train_on_text=True + train_on_pos=True)
            _pos  = batch["pos_ids"].to(device) if "pos_ids" in batch else None

            _optimizer.zero_grad()
            out = _model(_ids, _mask, pos_ids=_pos, labels=_bl,
                         intonation_labels=_il, break_idx_labels=_xl,
                         loss_weights=_loss_weights)
            losses = out["losses"]
            loss = sum(
                w * losses[k]
                for k, w in [("boundary", _BTW), ("intonation", _ITW), ("break_idx", _XTW)]
                if k in losses
            )
            loss.backward()
            nn.utils.clip_grad_norm_(_model.parameters(), max_norm=1.0)
            _optimizer.step()
            _scheduler.step()

            _train_loss += loss.item()
            t_b_logits.append(out["boundary_logits"].detach().cpu())
            t_b_labels.append(_bl.detach().cpu())
            t_i_logits.append(out["intonation_logits"].detach().cpu())
            t_i_labels.append(_il.detach().cpu())
            t_x_logits.append(out["break_idx_logits"].detach().cpu())
            t_x_labels.append(_xl.detach().cpu())
            t_b_spk_masks.append(batch["spk_mask"].cpu())

        _train_loss /= len(_train_loader)
        t_bm = compute_metrics(*flatten_predictions(t_b_logits, t_b_labels, all_spk_masks=t_b_spk_masks))
        t_im = compute_multiclass_metrics(*flatten_predictions(t_i_logits, t_i_labels))
        t_xm = compute_multiclass_metrics(*flatten_predictions(t_x_logits, t_x_labels))

        # ── Validate ───────────────────────────────────────────────────────
        _model.eval()
        _val_loss = 0.0
        v_b_logits, v_b_labels = [], []
        v_i_logits, v_i_labels = [], []
        v_x_logits, v_x_labels = [], []
        v_b_spk_masks = []

        with torch.no_grad():
            for batch in _val_loader:
                _ids  = batch["input_ids"].to(device)
                _mask = batch["attention_mask"].to(device)
                _bl   = batch["labels"].to(device)
                _il   = batch["intonation_labels"].to(device)
                _xl   = batch["break_idx_labels"].to(device)
                _pos  = batch["pos_ids"].to(device) if "pos_ids" in batch else None

                out = _model(_ids, _mask, pos_ids=_pos, labels=_bl,
                             intonation_labels=_il, break_idx_labels=_xl,
                             loss_weights=_loss_weights)
                losses = out["losses"]
                loss = sum(
                    w * losses[k]
                    for k, w in [("boundary", _BTW), ("intonation", _ITW), ("break_idx", _XTW)]
                    if k in losses
                )
                _val_loss += loss.item()
                v_b_logits.append(out["boundary_logits"].cpu())
                v_b_labels.append(_bl.cpu())
                v_i_logits.append(out["intonation_logits"].cpu())
                v_i_labels.append(_il.cpu())
                v_x_logits.append(out["break_idx_logits"].cpu())
                v_x_labels.append(_xl.cpu())
                v_b_spk_masks.append(batch["spk_mask"].cpu())

        _val_loss /= max(len(_val_loader), 1)
        v_bm = compute_metrics(*flatten_predictions(v_b_logits, v_b_labels, all_spk_masks=v_b_spk_masks))
        v_im = compute_multiclass_metrics(*flatten_predictions(v_i_logits, v_i_labels))
        v_xm = compute_multiclass_metrics(*flatten_predictions(v_x_logits, v_x_labels))

        # ── Log ────────────────────────────────────────────────────────────
        _history["train_loss"].append(_train_loss)
        _history["val_loss"].append(_val_loss)
        _history["train_boundary_f1"].append(t_bm["f1"])
        _history["val_boundary_f1"].append(v_bm["f1"])
        _history["train_boundary_prec"].append(t_bm["precision"])
        _history["val_boundary_prec"].append(v_bm["precision"])
        _history["train_boundary_rec"].append(t_bm["recall"])
        _history["val_boundary_rec"].append(v_bm["recall"])
        _history["train_intonation_f1"].append(t_im["f1"])
        _history["val_intonation_f1"].append(v_im["f1"])
        _history["train_break_idx_f1"].append(t_xm["f1"])
        _history["val_break_idx_f1"].append(v_xm["f1"])

        star = "★" if v_bm["f1"] > _best_val_f1 else " "
        print(f"Ep {epoch:02d}/{_EPOCHS}  loss={_train_loss:.4f}→{_val_loss:.4f}  "
              f"| boundary F1={t_bm['f1']:.4f}→{v_bm['f1']:.4f} {star}"
              f"| inton F1={v_im['f1']:.4f}  break F1={v_xm['f1']:.4f}")

        if v_bm["f1"] > _best_val_f1:
            _best_val_f1 = v_bm["f1"]
            _best_epoch  = epoch
            _best_state  = {k: v.cpu().clone() for k, v in _model.state_dict().items()}

    print("=" * 70)
    print(f"Training complete.  Best val boundary F1 = {_best_val_f1:.4f}  (epoch {_best_epoch})")

    # ── 9. Test evaluation ────────────────────────────────────────────────────
    _model.load_state_dict(_best_state)
    _model.eval()

    tb_logits, tb_labels = [], []
    ti_logits, ti_labels = [], []
    tx_logits, tx_labels = [], []
    tb_spk_masks = []
    per_sample = []

    with torch.no_grad():
        for batch in _test_loader:
            _ids  = batch["input_ids"].to(device)
            _mask = batch["attention_mask"].to(device)
            _bl   = batch["labels"].to(device)
            _il   = batch["intonation_labels"].to(device)
            _xl   = batch["break_idx_labels"].to(device)
            _pos  = batch["pos_ids"].to(device) if "pos_ids" in batch else None

            out     = _model(_ids, _mask, pos_ids=_pos)
            b_preds = out["boundary_logits"].argmax(dim=-1)
            i_preds = out["intonation_logits"].argmax(dim=-1)
            x_preds = out["break_idx_logits"].argmax(dim=-1)

            tb_logits.append(out["boundary_logits"]); tb_labels.append(_bl)
            tb_spk_masks.append(batch["spk_mask"])
            ti_logits.append(out["intonation_logits"]); ti_labels.append(_il)
            tx_logits.append(out["break_idx_logits"]); tx_labels.append(_xl)

            for sid, tokens, bp, bl, ip, il, xp, xl in zip(
                batch["sample_ids"], batch["tokens"],
                b_preds, _bl, i_preds, _il, x_preds, _xl,
            ):
                per_sample.append({
                    "sample_id":        sid,
                    "tokens":           tokens,
                    "boundary_preds":   bp[bl != -100].cpu().tolist(),
                    "boundary_true":    bl[bl != -100].cpu().tolist(),
                    "intonation_preds": ip[il != -100].cpu().tolist(),
                    "intonation_true":  il[il != -100].cpu().tolist(),
                    "break_idx_preds":  xp[xl != -100].cpu().tolist(),
                    "break_idx_true":   xl[xl != -100].cpu().tolist(),
                })

    fp_b, fl_b = flatten_predictions(tb_logits, tb_labels, all_spk_masks=tb_spk_masks)
    fp_i, fl_i = flatten_predictions(ti_logits, ti_labels)
    fp_x, fl_x = flatten_predictions(tx_logits, tx_labels)
    test_metrics = {
        "boundary":   compute_metrics(fp_b, fl_b),
        "intonation": compute_multiclass_metrics(fp_i, fl_i),
        "break_idx":  compute_multiclass_metrics(fp_x, fl_x),
    }

    print("=" * 55)
    print(f"  TEST SET  (best checkpoint — epoch {_best_epoch})")
    print("=" * 55)
    b = test_metrics["boundary"]
    print(f"  Boundary   F1={b['f1']:.4f}  P={b['precision']:.4f}  R={b['recall']:.4f}"
          f"  {'✓ beats baseline!' if b['f1'] > 0.77 else '✗ below 0.77 baseline'}")
    ii = test_metrics["intonation"]
    print(f"  Intonation F1={ii['f1']:.4f}  (macro, 3-class)")
    x = test_metrics["break_idx"]
    print(f"  Break idx  F1={x['f1']:.4f}  (macro, 2-class)")
    print("=" * 55)

    # ── 10. Plots ─────────────────────────────────────────────────────────────
    epochs_ax = list(range(1, _EPOCHS + 1))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.plot(epochs_ax, _history["train_loss"], label="Train")
    ax.plot(epochs_ax, _history["val_loss"],   label="Val", linestyle="--")
    ax.axvline(_best_epoch, color="grey", linestyle=":", linewidth=1,
               label=f"Best (ep {_best_epoch})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Combined loss")
    ax.set_title("Loss"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(epochs_ax, _history["train_boundary_f1"],  label="Train boundary F1")
    ax.plot(epochs_ax, _history["val_boundary_f1"],    label="Val boundary F1",   linestyle="--")
    ax.plot(epochs_ax, _history["val_intonation_f1"],  label="Val intonation F1", linestyle=":")
    ax.plot(epochs_ax, _history["val_break_idx_f1"],   label="Val break idx F1",  linestyle="-.")
    ax.axhline(0.77, color="red", linestyle=":", linewidth=1.2, label="Baseline F1 = 0.77")
    ax.axvline(_best_epoch, color="grey", linestyle=":", linewidth=1)
    ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
    ax.set_title("F1 by head"); ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 1)

    fig.suptitle(run_id, fontsize=8, color="grey")
    plt.tight_layout()
    plt.savefig(f"/tmp/{run_id}_curves.png", dpi=120, bbox_inches="tight")
    plt.show()

    cm = confusion_matrix(fl_b, fp_b, labels=[0, 1])
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=["non-boundary", "boundary"]).plot(
        ax=ax2, colorbar=False, cmap="Blues", values_format="d")
    ax2.set_title(f"Boundary — epoch {_best_epoch}\n{run_id}", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"/tmp/{run_id}_confusion_matrix.png", dpi=120, bbox_inches="tight")
    plt.show()

    # ── 11. Save to Drive ─────────────────────────────────────────────────────
    os.makedirs(run_dir, exist_ok=True)
    ckpt_dir = os.path.join(run_dir, "checkpoint")
    _model.save_pretrained(ckpt_dir)
    _tokenizer.save_pretrained(ckpt_dir)

    hparam_record = {
        "run_id": run_id, "run_notes": _NOTES,
        "timestamp": datetime.now().isoformat(),
        "data": {
            "n_samples_total": len(_samples),
            "n_train": len(_train_ds), "n_val": len(_val_ds), "n_test": len(_test_ds),
            "split_seed": _SPLIT_SEED, "train_frac": _TRAIN_FRAC, "val_frac": _VAL_FRAC,
        },
        "preprocessing": {
            "strip_punctuation": _STRIP_PUNCT,
            "train_on_text": _TRAIN_ON_TEXT,
            "train_on_pos":  _TRAIN_ON_POS,
            "pos_embedding_dim": _POS_EMB_DIM if _use_pos_emb else None,
            "extra_features": _EXTRA_FEATURES, "max_length": _MAX_LENGTH,
        },
        "model": {"base": _MODEL_NAME, "use_pos_embedding": _use_pos_emb},
        "training": {
            "batch_size": _BATCH_SIZE, "learning_rate": _LR,
            "num_epochs": _EPOCHS, "warmup_steps": _WARMUP, "weight_decay": _WD,
            "imbalance_strategy": _IMBALANCE, "boundary_loss_weight": _BLW,
            "boundary_task_weight": _BTW, "intonation_task_weight": _ITW,
            "break_idx_task_weight": _XTW, "best_epoch": _best_epoch,
        },
        "results": {
            "best_val_boundary_f1": _best_val_f1,
            "test": test_metrics,
            "val_history": {
                "loss":          _history["val_loss"],
                "boundary_f1":   _history["val_boundary_f1"],
                "boundary_prec": _history["val_boundary_prec"],
                "boundary_rec":  _history["val_boundary_rec"],
                "intonation_f1": _history["val_intonation_f1"],
                "break_idx_f1":  _history["val_break_idx_f1"],
            },
            "train_history": {
                "loss":          _history["train_loss"],
                "boundary_f1":   _history["train_boundary_f1"],
                "intonation_f1": _history["train_intonation_f1"],
                "break_idx_f1":  _history["train_break_idx_f1"],
            },
        },
        "annotation_tool_meta": label_meta,
    }
    with open(os.path.join(run_dir, f"{run_id}_hparams.json"), "w") as f:
        json.dump(hparam_record, f, indent=2, cls=_NumpyEncoder)
    with open(os.path.join(run_dir, f"{run_id}_test_predictions.json"), "w") as f:
        json.dump(per_sample, f, indent=2)
    for tag in ["curves", "confusion_matrix"]:
        shutil.copy(f"/tmp/{run_id}_{tag}.png", os.path.join(run_dir, f"{run_id}_{tag}.png"))

    print(f"\n✓ Saved → {run_dir}")

    # ── 12. Clear GPU memory before next run ──────────────────────────────────
    _model.cpu()
    del _model, _optimizer, _scheduler, _best_state
    del _train_ds, _val_ds, _test_ds
    del _train_loader, _val_loader, _test_loader
    gc.collect()
    torch.cuda.empty_cache()
    subprocess.run(["sync"], check=False)
    print(f"✓ GPU memory cleared.  Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    return {"run_id": run_id, "run_dir": run_dir, "test_metrics": test_metrics}


print("✓ run_experiment() defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Run queue                                         ║
# ║  Edit RUNS list below to choose which experiments to run.    ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Load data sources ─────────────────────────────────────────────────────────
samples_360, _    = load_label_files_batched(BATCHED_ROOT_360)
samples_libritts  = all_samples + samples_360
print(f"✓ LibriTTS (c100+c360): {len(samples_libritts):,} samples")

samples_sbc, _    = load_label_files_batched(BATCHED_ROOT_SBC)
print(f"✓ SBCSAE:               {len(samples_sbc):,} samples")

samples_all = samples_libritts + samples_sbc
print(f"✓ Combined (all):       {len(samples_all):,} samples")

# ── POS enrichment (only when TRAIN_ON_POS=True) ──────────────────────────────
# Generate POS files for c360 and SBC if needed, then enrich all sample lists.
# clean-100 (all_samples) was already enriched in Cell 4b.
# POS files are written once and reused; this block is cheap after first run.
if TRAIN_ON_POS:
    print("\nGenerating / loading POS for clean-360 ...")
    _generate_pos_files(BATCHED_ROOT_360)
    _load_pos_for_samples(samples_360, BATCHED_ROOT_360)

    print("\nGenerating / loading POS for SBCSAE ...")
    _generate_pos_files(BATCHED_ROOT_SBC)
    _load_pos_for_samples(samples_sbc, BATCHED_ROOT_SBC)

    # Rebuild combined lists now that all sublists have pos_tags
    samples_libritts = all_samples + samples_360   # both already enriched
    samples_all      = samples_libritts + samples_sbc
    print(f"\n✓ POS enrichment complete for all {len(samples_all):,} samples.")

# ── Loss weight ───────────────────────────────────────────────────────────────
# Update from Cell 4's printed ratio suggestion after loading.
AUTO_LOSS_WEIGHT = 4.0

# ── Run queue ─────────────────────────────────────────────────────────────────
# SBCSAE-only baselines (no POS, always run)
RUNS = [
]

# ── POS experiments (appended only when TRAIN_ON_POS=True) ────────────────────
# Four runs: 2 combined (text+POS) × 2 punctuation settings
#          + 2 POS-only × 2 punctuation settings
# All use the full corpus (LibriTTS silver + SBCSAE gold).
# SBC001-005 is the held-out test set for all of these (matching the baseline).
if TRAIN_ON_POS:
    RUNS += [
        # ── Combined: text + POS embedding, without punctuation ───────────
        {
            "overrides": {
                "STRIP_PUNCTUATION": True,
                "TRAIN_ON_TEXT":     True,
                "TRAIN_ON_POS":      True,
                "RUN_NOTES":         "libri+sbc_text+pos_stl",
            },
            "samples": samples_all,
        },
    ]

# ── Execute all runs ──────────────────────────────────────────────────────────
results_summary = []
for i, run_def in enumerate(RUNS):
    print(f"\n{'='*70}\n  STARTING RUN {i+1} of {len(RUNS)}\n{'='*70}")
    result = run_experiment(run_def["overrides"], samples_override=run_def["samples"])
    results_summary.append(result)

print("\n\n" + "█"*70)
print("  ALL RUNS COMPLETE — SUMMARY")
print("█"*70)
print(f"  {'Run ID':<45}  {'Boundary F1':>12}  {'Inton F1':>10}  {'Break F1':>10}")
print("  " + "-"*83)
for r in results_summary:
    b  = r["test_metrics"]["boundary"]["f1"]
    iv = r["test_metrics"]["intonation"]["f1"]
    x  = r["test_metrics"]["break_idx"]["f1"]
    flag = "✓" if b > 0.77 else "✗"
    print(f"  {r['run_id']:<45}  {b:>11.4f}{flag}  {iv:>10.4f}  {x:>10.4f}")
print("█"*70)


In [ ]:
'''
    # ── PRIORITY 2: SBCSAE only, standard loss ────────────────────────────────
    {
        "overrides": {
            "STRIP_PUNCTUATION": True,
            "IMBALANCE_STRATEGY": "none",
            "RUN_NOTES": "np_sbcsae_stloss",
        },
        "samples": samples_sbc,
    },
'''
